<h3 style="text-align: center;"><b>Школа глубокого обучения ФПМИ МФТИ</b></h3>

<h3 style="text-align: center;"><b>Домашнее задание. Детекция объектов</b></h3>

В этом домашнем задании мы продолжим работу над детектором из семинара, поэтому при необходимости можете заимствовать оттуда любой код.

Домашнее задание можно разделить на следующие части:

* Переделываем модель [4]
  * Backbone[1],
  * Neck [2],
  * Head [1]
* Label assignment [3]:
  * TAL [3]
* Лоссы [1]:
  * CIoU loss [1]
* Кто больше? [5]
  * 0.05 mAP [1]
  * 0.1 mAP  [2]
  * 0.2 mAP [5]

**Максимальный балл:** 10 баллов. (+3 балла бонус).

In [ ]:
import torch
import numpy as np
import pandas as pd
import albumentations as A

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset
from albumentations.pytorch.transforms import ToTensorV2
import io
import math
from tqdm.auto import tqdm

import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.ops import box_iou, nms, distance_box_iou_loss

### Загрузка данных

Мы продолжаем работу с датасетом из семинара - Halo infinite ([сслыка](https://universe.roboflow.com/graham-doerksen/halo-infinite-angel-aim)). Загрузка данных и создание датасета полностью скопированы из семинара.

Сначала загружаем данные

In [ ]:
splits = {'train': 'data/train-00000-of-00001-0d6632d599c29801.parquet',
          'validation': 'data/validation-00000-of-00001-c6b77a557eeedd52.parquet',
          'test': 'data/test-00000-of-00001-866d29d8989ea915.parquet'}
df_train = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["train"])
df_test = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["test"])

Создаем датасет для предобработки данных

In [ ]:
class HaloDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        df_objects = pd.json_normalize(dataframe['objects'])[["bbox", "category"]]
        df_images = pd.json_normalize(dataframe['image'])[["bytes"]]
        self.data = dataframe[["image_id"]].join(df_objects).join(df_images)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """Загружаем данные и разметку для объекта с индексом `idx`.

        labels: List[int] Набор классов для каждого ббокса,
        boxes: List[List[int]] Набор ббоксов в формате (x_min, y_min, w, h).
        """
        row = self.data.iloc[idx]
        image = Image.open(io.BytesIO(row["bytes"]))
        image = np.array(image)

        target = {}
        target["image_id"] = row["image_id"]

        labels = [row["category"]] if isinstance(row["category"], int) else row['category']
        # Вычитаем единицу чтобы классы начинались с нуля
        labels = [label - 1 for label in labels]
        boxes = row['bbox'].tolist()

        if self.transform is not None:
            transformed = self.transform(image=image, bboxes=boxes, labels=labels)
            image, boxes, labels = transformed["image"], transformed["bboxes"], transformed["labels"]
        else:
            image = transforms.ToTensor()(image)

        # COCO: x_min, y_min, w, h -> XYXY: x_min, y_min, x_max, y_max
        boxes = torch.tensor(np.array(boxes), dtype=torch.float32)
        if len(boxes) > 0:
            boxes[:, 2] = boxes[:, 0] + boxes[:, 2]
            boxes[:, 3] = boxes[:, 1] + boxes[:, 3]
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)

        target['boxes'] = boxes
        target['labels'] = torch.tensor(labels, dtype=torch.int64)
        return image, target

def collate_fn(batch):
    batch = tuple(zip(*batch))
    images = torch.stack(batch[0])
    return images, batch[1]

Чтобы модель не переобучалась, можно добавить больше аугментаций, весь список можно посмотреть тут [[ссылка](https://explore.albumentations.ai/)].

Какие можно использовать аугментации?
* Добавить зум `RandomResizedCrop`,
* Сделать цветовые аугментации типа `RandomBrightnessContrast` и/или `HueSaturationValue`,
* Добавить шум `GaussNoise`,
* Вырезать случайные части изображения `CoarseDropout`,
* И любые другие!

Аугментации можно комбинировать посредствам `A.OneOf`, `A.SomeOf` или `A.RandomOrder`.

Хоть аугментации ограничиваются только вашей фантазией, перед обучением советуем посмотреть на результат преобразований и убедиться, что изображение ещё поддается детекции:)

In [ ]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_transform = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.HueSaturationValue(p=0.2),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(format='coco', label_fields=['labels'])
)

test_transform = A.Compose(
    [
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(format='coco', label_fields=['labels'])
)

Не забываем инициализировать наш датасет

In [ ]:
train_dataset = HaloDataset(df_train, transform=train_transform)
test_dataset = HaloDataset(df_test, transform=test_transform)

In [ ]:
batch_size = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

all_labels = []

for objects in df_train["objects"]:
    cats = objects["category"]

    if isinstance(cats, np.ndarray):
        cats = cats.tolist()

    if isinstance(cats, list):
        all_labels.extend(cats)
    else:
        all_labels.append(cats)

all_labels = [int(x) for x in all_labels]

NUM_CLASSES = max(all_labels) + 1

print("NUM_CLASSES:", NUM_CLASSES)
print("labels:", sorted(set(all_labels)))

## Переделываем модель [4 балла]

В семинаре мы реализовали самый базовый детектор, а сейчас настало время его улучшать.

### Backbone [1 балл]

Хорошей практикой считается размораживать несколько последних слоев в backbone, это позволяет немного улучить качество модели. Давайте улушчим класс Backbone из лекции, добавив ему возможность разморозки __k__ последних слоев или блоков (на ваш выбор).

In [ ]:
class Backbone(nn.Module):
    def __init__(self, pretrained=True, unfreeze_last=2):
        super().__init__()

        weights = ResNet18_Weights.DEFAULT if pretrained else None
        model = resnet18(weights=weights)

        self.stem = nn.Sequential(
            model.conv1,
            model.bn1,
            model.relu,
            model.maxpool
        )

        self.layers = nn.ModuleList([
            model.layer1,
            model.layer2,
            model.layer3,
            model.layer4
        ])

        self.out_channels = [64, 128, 256, 512]

        # Сначала замораживаем весь backbone
        for param in self.parameters():
            param.requires_grad = False

        # Размораживаем последние k блоков
        if unfreeze_last > 0:
            for layer in self.layers[-unfreeze_last:]:
                for param in layer.parameters():
                    param.requires_grad = True

    def forward(self, x):
        x = self.stem(x)

        features = []
        for layer in self.layers:
            x = layer(x)
            features.append(x)

        return features

### NECK [2 балла]

Следующее улучшение коснется шеи. Предлагаем реализовать знакомую из лекции архитектуру FPN.

#### Feature Pyramid Network

<center><img src="https://user-images.githubusercontent.com/57972646/69858594-b14a6c00-12d5-11ea-8c3e-3c17063110d3.png"/></center>


* [Feature Pyramid Networks for Object Detection](https://arxiv.org/abs/1612.03144)

Она состоит из top-down пути, в котором происходит 2 вещи:
1. Увеличивается пространственная размерность фичей,
2. С помощью скипконнекшеннов, добавляются фичи из backbone модели.

Для увеличения пространственной размерности используется __nearest neighbor upsampling__, а фичи из шеи и бекбоуна суммируются.

__TIPS__:
* Можете использовать базовые классы из лекции,
* Воспользуйтесь AnchorGenerator-ом, чтобы создавать якоря сразу для нескольких выходов,
* Не забудьте использовать nn.ModuleList, если захотите сделать динамическое количество голов у модели,
* Также, можно добавить доп конволюцию (3х3 с паддингом) у каждого выхода шеи.

In [ ]:
import torch.nn as nn

class Neck(nn.Module):
    def __init__(self, in_channels=(64, 128, 256, 512), out_channels=128):
        super().__init__()

        self.lateral_convs = nn.ModuleList([
            nn.Conv2d(c, out_channels, kernel_size=1)
            for c in in_channels
        ])

        self.output_convs = nn.ModuleList([
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
            for _ in in_channels
        ])

        self.out_channels = out_channels

    def forward(self, features):
        laterals = [conv(f) for conv, f in zip(self.lateral_convs, features)]

        # top-down FPN
        for i in range(len(laterals) - 1, 0, -1):
            upsampled = F.interpolate(
                laterals[i],
                size=laterals[i - 1].shape[-2:],
                mode="nearest"
            )
            laterals[i - 1] = laterals[i - 1] + upsampled

        outputs = [conv(f) for conv, f in zip(self.output_convs, laterals)]
        return outputs

### Head [1 балл]

В качестве шеи можно выбрать __один из двух__ вариантов:

#### 1. Decoupled Head

Реализовать Decoupled Head из [YOLOX](https://arxiv.org/abs/2107.08430).
<center><img src="https://i.ibb.co/BVtBR2R3/Decoupled-head.jpg"/></center>

**TIP**: Возьмите за основу голову из семинара, тк она сильно похожа на Decoupled Head.

Изменять количество параметров у шей на разных уровнях не обязательно.

#### 2. Confidence score free head

Нужно взять за основу голову из семинара и полностью убрать предсказание confidence score. Чтобы модель предсказывала только 2 группы: ббоксы и классы.

Есть следующие способы удаления confidence score:
* Добавление нового класса ФОН. Обычно его обозначают нулевым классом.
* Присваивание ббоксам БЕЗ объекта вектор из нулей в качестве таргета.

Выберете тот, который вам больше нравится и будте внимательны при расчете лосса!

**Важно!** Удаление confidence score повлияет на следующие методы из семинара:
* target_assign
* ComputeLoss
* _filter_predictions

In [ ]:
class Head(nn.Module):
    def __init__(self, in_channels=128, num_classes=NUM_CLASSES):
        super().__init__()

        self.num_classes = num_classes

        self.stem = nn.Conv2d(in_channels, in_channels, kernel_size=1)

        self.cls_branch = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

        self.reg_branch = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

        self.cls_pred = nn.Conv2d(in_channels, num_classes, kernel_size=1)
        self.obj_pred = nn.Conv2d(in_channels, 1, kernel_size=1)
        self.box_pred = nn.Conv2d(in_channels, 4, kernel_size=1)

    def forward_single(self, x):
        x = self.stem(x)

        cls_feat = self.cls_branch(x)
        reg_feat = self.reg_branch(x)

        cls_logits = self.cls_pred(cls_feat)
        obj_logits = self.obj_pred(reg_feat)
        box_reg = self.box_pred(reg_feat)

        return cls_logits, obj_logits, box_reg

    def forward(self, features):
        outputs = []

        for feature in features:
            cls_logits, obj_logits, box_reg = self.forward_single(feature)
            outputs.append({
                "cls_logits": cls_logits,
                "obj_logits": obj_logits,
                "box_reg": box_reg
            })

        return outputs

Теперь можно снова реализовать класс детектора с учетом всех частей выше!

In [ ]:
class Detector(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, fpn_channels=128, unfreeze_last=2):
        super().__init__()

        self.backbone = Backbone(pretrained=True, unfreeze_last=unfreeze_last)
        self.neck = Neck(in_channels=self.backbone.out_channels, out_channels=fpn_channels)
        self.head = Head(in_channels=fpn_channels, num_classes=num_classes)

        self.num_classes = num_classes
        self.anchor_sizes = [16, 32, 64, 128]

    def make_anchors(self, feature, image_h, image_w, size):
        _, _, h, w = feature.shape
        device = feature.device

        stride_y = image_h / h
        stride_x = image_w / w

        shifts_x = (torch.arange(w, device=device) + 0.5) * stride_x
        shifts_y = (torch.arange(h, device=device) + 0.5) * stride_y

        yy, xx = torch.meshgrid(shifts_y, shifts_x, indexing="ij")

        cx = xx.reshape(-1)
        cy = yy.reshape(-1)

        half = size / 2

        anchors = torch.stack([
            cx - half,
            cy - half,
            cx + half,
            cy + half
        ], dim=1)

        return anchors

    def decode_boxes(self, anchors, box_reg):
        # box_reg: N x 4
        ax1, ay1, ax2, ay2 = anchors.unbind(dim=1)

        acx = (ax1 + ax2) / 2
        acy = (ay1 + ay2) / 2
        aw = ax2 - ax1
        ah = ay2 - ay1

        dx, dy, dw, dh = box_reg.unbind(dim=1)

        pred_cx = acx + dx.tanh() * aw
        pred_cy = acy + dy.tanh() * ah
        pred_w = aw * torch.exp(dw.clamp(min=-4, max=4))
        pred_h = ah * torch.exp(dh.clamp(min=-4, max=4))

        x1 = pred_cx - pred_w / 2
        y1 = pred_cy - pred_h / 2
        x2 = pred_cx + pred_w / 2
        y2 = pred_cy + pred_h / 2

        return torch.stack([x1, y1, x2, y2], dim=1)

    def forward(self, images):
        image_h, image_w = images.shape[-2:]

        features = self.backbone(images)
        features = self.neck(features)
        head_outputs = self.head(features)

        all_boxes = []
        all_cls_logits = []
        all_obj_logits = []
        all_anchors = []

        for level, output in enumerate(head_outputs):
            cls_logits = output["cls_logits"]
            obj_logits = output["obj_logits"]
            box_reg = output["box_reg"]

            b, _, h, w = cls_logits.shape

            cls_logits = cls_logits.permute(0, 2, 3, 1).reshape(b, -1, self.num_classes)
            obj_logits = obj_logits.permute(0, 2, 3, 1).reshape(b, -1)
            box_reg = box_reg.permute(0, 2, 3, 1).reshape(b, -1, 4)

            anchors = self.make_anchors(
                output["box_reg"],
                image_h=image_h,
                image_w=image_w,
                size=self.anchor_sizes[level]
            )

            decoded = []
            for i in range(b):
                decoded.append(self.decode_boxes(anchors, box_reg[i]))

            decoded = torch.stack(decoded, dim=0)

            all_boxes.append(decoded)
            all_cls_logits.append(cls_logits)
            all_obj_logits.append(obj_logits)
            all_anchors.append(anchors)

        return {
            "boxes": torch.cat(all_boxes, dim=1),
            "cls_logits": torch.cat(all_cls_logits, dim=1),
            "obj_logits": torch.cat(all_obj_logits, dim=1),
            "anchors": torch.cat(all_anchors, dim=0)
        }

## Label assignment [3 балла]
В этой секции предлагается заменить функцию `assign_target` на более современный алгоритм который называется Task alignment learning.

Он описан в статье [TOOD](https://arxiv.org/abs/2108.07755) в секции 3.2. Для удобства вот его основные шаги:

1. Посчитать значение метрики для каждого предсказанного ббокса:
    
$$t = s^\alpha * u^\beta$$
    
где,
* $s$ — classification score, или вероятность принадлежности предсказанного ббокса к классу реального ббокса (**GT**);
* $u$ — IoU между предсказанным и реальным ббоксами;
* $\alpha,\ \beta$ — нормализационные константы, обычно $\alpha = 6.0, \ \beta = 1.0$.
    
2. Отфильтровать предсказания на основе **GT**.

    Для якорных детекторов, обычно, выбираются только те предсказания, центры якорей которых находятся внутри GT.
4. Для каждого **GT** выбрать несколько (обычно 5 или 13) самых подходящих предсказаний.
5. Если предсказание рассматривается в качестве подходящего для нескольких **GT** — выбрать **GT** с наибольшим пересечением по IoU.


**BAЖНО**: если будете использовать Runner из лекции, не забудьте поменять параметры  в `self.assign_target_method` в методе `_run_train_epoch`.

In [ ]:
def TAL_assigner(pred_boxes, cls_logits, gt_boxes, gt_labels, anchors=None, topk=13, alpha=6.0, beta=1.0):
    device = pred_boxes.device
    num_preds = pred_boxes.shape[0]

    target_boxes = torch.zeros((num_preds, 4), device=device)
    target_labels = torch.full((num_preds,), -1, dtype=torch.long, device=device)
    fg_mask = torch.zeros((num_preds,), dtype=torch.bool, device=device)

    if gt_boxes.numel() == 0:
        return target_boxes, target_labels, fg_mask

    gt_boxes = gt_boxes.to(device)
    gt_labels = gt_labels.to(device)

    num_gt = gt_boxes.shape[0]

    ious = box_iou(pred_boxes, gt_boxes)  # N x M

    cls_scores = cls_logits.sigmoid()
    gt_scores = cls_scores[:, gt_labels]  # N x M

    alignment_metric = (gt_scores ** alpha) * (ious ** beta)

    if anchors is not None:
        anchors = anchors.to(device)
        centers_x = (anchors[:, 0] + anchors[:, 2]) / 2
        centers_y = (anchors[:, 1] + anchors[:, 3]) / 2

        inside_gt = (
            (centers_x[:, None] >= gt_boxes[None, :, 0]) &
            (centers_y[:, None] >= gt_boxes[None, :, 1]) &
            (centers_x[:, None] <= gt_boxes[None, :, 2]) &
            (centers_y[:, None] <= gt_boxes[None, :, 3])
        )

        alignment_metric = alignment_metric * inside_gt.float()

    candidate_mask = torch.zeros_like(alignment_metric, dtype=torch.bool)

    for gt_idx in range(num_gt):
        metric_for_gt = alignment_metric[:, gt_idx]

        k = min(topk, num_preds)
        _, topk_idx = torch.topk(metric_for_gt, k=k)

        candidate_mask[topk_idx, gt_idx] = True

    candidate_ious = ious.clone()
    candidate_ious[~candidate_mask] = -1

    best_ious, matched_gt_idx = candidate_ious.max(dim=1)

    fg_mask = best_ious > 0

    if fg_mask.any():
        target_boxes[fg_mask] = gt_boxes[matched_gt_idx[fg_mask]]
        target_labels[fg_mask] = gt_labels[matched_gt_idx[fg_mask]]

    return target_boxes, target_labels, fg_mask

### DIoU [1]

Вместо SmoothL1, который используется в семинаре, реализуем лосс, основанный на пересечении ббоксов. В качестве тренировки давайте напишем Distance Intersection over Union (DIoU).

<center><img src=https://wikidocs.net/images/page/163613/Free_Fig_5.png></center>

Для его реализации разобъем задачу на части:

**1. Реализуем IoU:**

Пусть даны координаты для предсказанного ($B^p$) и истинного ($B^g$) ббоксов в формате XYXY или VOC PASCAL (левый верхний и правый нижний углы):

$B^p=(x^p_1, y^p_1, x^p_2, y^p_2)$, $B^g=(x^g_1, y^g_1, x^g_2, y^g_2)$, тогда алгоритм расчета будет следующий:

    1. Найдем площади обоих ббоксов:
$$ A^p = (x^p_2 - x^p_1) * (y^p_2 - y^p_1) $$
$$ A^g = (x^g_2 - x^g_1) * (y^g_2 - y^g_1) $$

    2. Посчитаем пересечение между ббоксами:

Тут мы предлагаем вам подумать как в общем виде можно расчитать размеры ббокса, который будет являться пересечением $B^p$ и $B^g$, а затем посчитать его площадь:

$$x^I_1 = \qquad \qquad y^I_1 = $$
$$x^I_2 = \qquad \qquad y^I_2 = $$

В общем виде, площать будет записываться следующим образом:

Если $x^I_2 > x^I_1$ & $y^I_2 > y^I_1$, тогда:

$$I = (x^I_2 - x^I_1) * (y^I_2 - y^I_1)$$

Иначе, $I = 0$.

    3. Считаем объединение ббоксов.

Мы можем посчитать эту площадь как сумму площадей двух ббоксов минус площадь пересечения (тк мы считаем её два раз в сумме площадей):

$$U = A^p + A^g - I$$

    4. Вычисляем IoU.

$$IoU = \frac{I}{U}$$

**2. Посчитаем диагональ выпуклой оболочки:**

Для расчета диагонали, сначала выпишите координаты верхнего левого и правого нижнего углов. Подумайте, чему будут равны эти координаты в общем случае?

$$x^c_1 = \qquad \qquad y^c_1 = $$
$$x^c_2 = \qquad \qquad y^c_2 = $$

Подсказка: Нарисуйте несколько вариантов пересечений предсказания и GT на бумажке, и выпишите координаты для выпуклой оболочки.

Тогда квадрат диагонали можно посчитать по формуле:

$$c^2 = (x^c_2 - x^c_1)^2 + (y^c_2 - y^c_1)^2$$

**3. Рассчитаем расстояние между цетрами ббоксов:**

Сначала находим координаты центров каждого из ббоксов (если ббоксы в формате YOLO, то и считать ничего не нужно), затем считаем Евклидово расстояние между центрами.

$d = $

Собираем все части вместе и считаем лосс по формуле:

$$ DIoU = 1 - IoU + \frac{d^2}{c^2}$$

Помните, что пар ббоксов может быть много! Возвращайте усредненное значение лосса.

In [ ]:
from torchvision.ops import distance_box_iou_loss

In [ ]:
def gen_bbox(num_boxes=10):
    min_corner = torch.randint(0, 100, (num_boxes, 2))
    max_corner = torch.randint(50, 150, (num_boxes, 2))

    for i in range(2):
        wrong_order = min_corner[:, i] > max_corner[:, i]
        if wrong_order.any():
            min_corner[wrong_order, i], max_corner[wrong_order, i] = max_corner[wrong_order, i], min_corner[wrong_order, i]
    return torch.cat((min_corner, max_corner), dim=1)

In [ ]:
pred_boxes = gen_bbox(num_boxes=100)
true_boxes = gen_bbox(num_boxes=100)

In [ ]:
print(f"DIoU: {distance_box_iou_loss(pred_boxes, true_boxes, reduction='mean').item()}")

In [ ]:
def diou_loss(pred_boxes, gt_boxes):
    eps = 1e-7

    pred_boxes = pred_boxes.float()
    gt_boxes = gt_boxes.float()

    # площади боксов
    pred_area = (pred_boxes[:, 2] - pred_boxes[:, 0]).clamp(min=0) * \
                (pred_boxes[:, 3] - pred_boxes[:, 1]).clamp(min=0)

    gt_area = (gt_boxes[:, 2] - gt_boxes[:, 0]).clamp(min=0) * \
              (gt_boxes[:, 3] - gt_boxes[:, 1]).clamp(min=0)

    # пересечение
    inter_x1 = torch.max(pred_boxes[:, 0], gt_boxes[:, 0])
    inter_y1 = torch.max(pred_boxes[:, 1], gt_boxes[:, 1])
    inter_x2 = torch.min(pred_boxes[:, 2], gt_boxes[:, 2])
    inter_y2 = torch.min(pred_boxes[:, 3], gt_boxes[:, 3])

    inter_w = (inter_x2 - inter_x1).clamp(min=0)
    inter_h = (inter_y2 - inter_y1).clamp(min=0)

    intersection = inter_w * inter_h

    # объединение
    union = pred_area + gt_area - intersection + eps
    iou = intersection / union

    # центры боксов
    pred_cx = (pred_boxes[:, 0] + pred_boxes[:, 2]) / 2
    pred_cy = (pred_boxes[:, 1] + pred_boxes[:, 3]) / 2

    gt_cx = (gt_boxes[:, 0] + gt_boxes[:, 2]) / 2
    gt_cy = (gt_boxes[:, 1] + gt_boxes[:, 3]) / 2

    center_distance = (pred_cx - gt_cx) ** 2 + (pred_cy - gt_cy) ** 2

    # диагональ минимального внешнего прямоугольника
    enc_x1 = torch.min(pred_boxes[:, 0], gt_boxes[:, 0])
    enc_y1 = torch.min(pred_boxes[:, 1], gt_boxes[:, 1])
    enc_x2 = torch.max(pred_boxes[:, 2], gt_boxes[:, 2])
    enc_y2 = torch.max(pred_boxes[:, 3], gt_boxes[:, 3])

    diagonal = (enc_x2 - enc_x1) ** 2 + (enc_y2 - enc_y1) ** 2 + eps

    loss = 1 - iou + center_distance / diagonal

    return loss.mean()

In [ ]:
import numpy as np
pred_boxes = gen_bbox(num_boxes=1000)
true_boxes = gen_bbox(num_boxes=1000)

# проверим что написанный лосс выдает те же результаты что и лосс из торча.
assert np.isclose(diou_loss(pred_boxes, true_boxes), distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean"))

## Кто больше? [5 баллов]

Наконец то мы дошли до самый интересной части. Тут мы раздаем очки за mAP'ы!

Все что вы написали выше вам поможет улучшить качество итогового детектора, настало время узнать насколько сильно :)

За достижения порога по mAP на тестовом наборе вы получаете баллы:
* 0.05 mAP [1]
* 0.1 mAP [2]
* 0.2 mAP [5]


**TIPS**:
1. На семинаре мы специально не унифицировали формат ббоксов между методами, чтобы обратить ваше внимание что за этим нужно следить. Чтобы было проще, сразу унифицируете формат по всему ноутбуку. Советуем использовать формат xyxy, тк IoU и NMS из torch используют именно этот формат. (Не забудьте поменять формат у таргета в `HaloDataset`).

2. Попробуйте перейти к IoU-based лоссу при обучении. То есть обучать не смещения, а сразу предсказывать ббокс.

3. Поэксперементируйте с подходами target assignment'а в процессе обучения. Например, можно на первых итерациях использовать обычный метод, а затем подключить TAL.

4. Добавьте аугментаций!

Можно взять [albumentations](https://albumentations.ai/docs/getting_started/bounding_boxes_augmentation/), библиотеку, которую мы использовали всеминаре. Или базовые аугментации из торча [тык](https://pytorch.org/vision/main/transforms.html). Если будете использовать торч, не забудте про ббоксы, transforms из коробки не будет их агументировать.

5. Можете реализовать другую шею, которую мы обсуждали на лекции [Path Aggregation Network](https://arxiv.org/abs/1803.01534) она точно улучшит ваше итоговое качество.

6. Попробуйте добавлять различные блоки из YOLO архитектур в шею вместо единичных конволюционных слоев. (Например, замените конволюции 3х3 на CSP блоки).

7. Попробуйте заменить NMS на другой метод (WeightedNMS, SoftNMS, etc.). Немного ссылок:
    * Статья про SoftNMS [тык](https://arxiv.org/pdf/1704.04503)
    * Статья про WeightedNMS [тык](https://openaccess.thecvf.com/content_ICCV_2017_workshops/papers/w14/Zhou_CAD_Scale_Invariant_ICCV_2017_paper.pdf)
    * Есть их реализация, правда на нумбе [git](https://github.com/ZFTurbo/Weighted-Boxes-Fusion?tab=readme-ov-file)

8. Не бойтесь эксперементировать и удачи!

Также, напишите развернутые ответы на следующие вопросы:

**Questions:**
1. Какой метод label assignment'a помогает лучше обучаться модели? Почему?
2. Какое из сделаных вами улучшений внесло наибольший вклад в качество модели? Как вы думаете, почему это произошло?
3. Какое из сделанных вами улучшений вообще не изменило метрику? Как вы думаете, почему это произошло?

Loss для обучения

In [ ]:
class DetectionLoss(nn.Module):
    def __init__(self, alpha=6.0, beta=1.0, topk=13):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.topk = topk

    def forward(self, outputs, targets):
        boxes = outputs["boxes"]
        cls_logits = outputs["cls_logits"]
        obj_logits = outputs["obj_logits"]
        anchors = outputs["anchors"]

        total_loss = 0

        for i in range(boxes.shape[0]):
            pred_boxes = boxes[i]
            pred_cls = cls_logits[i]
            pred_obj = obj_logits[i]

            gt_boxes = targets[i]["boxes"].to(pred_boxes.device)
            gt_labels = targets[i]["labels"].to(pred_boxes.device)

            target_boxes, target_labels, fg_mask = TAL_assigner(
                pred_boxes=pred_boxes,
                cls_logits=pred_cls,
                gt_boxes=gt_boxes,
                gt_labels=gt_labels,
                anchors=anchors,
                topk=self.topk,
                alpha=self.alpha,
                beta=self.beta
            )

            obj_target = fg_mask.float()

            obj_loss = F.binary_cross_entropy_with_logits(pred_obj, obj_target)

            if fg_mask.any():
                box_loss = diou_loss(pred_boxes[fg_mask], target_boxes[fg_mask])

                cls_target = torch.zeros_like(pred_cls)
                cls_target[fg_mask, target_labels[fg_mask]] = 1.0

                cls_loss = F.binary_cross_entropy_with_logits(
                    pred_cls[fg_mask],
                    cls_target[fg_mask]
                )
            else:
                box_loss = torch.tensor(0.0, device=pred_boxes.device)
                cls_loss = torch.tensor(0.0, device=pred_boxes.device)

            total_loss = total_loss + obj_loss + cls_loss + box_loss

        return total_loss / boxes.shape[0]


Фильтрация предсказаний для mAP

In [ ]:
def filter_predictions(outputs, score_threshold=0.1, nms_threshold=0.5):
    boxes = outputs["boxes"]
    cls_logits = outputs["cls_logits"]
    obj_logits = outputs["obj_logits"]

    batch_predictions = []

    for i in range(boxes.shape[0]):
        pred_boxes = boxes[i]
        cls_scores = cls_logits[i].sigmoid()
        obj_scores = obj_logits[i].sigmoid()

        scores_per_class = cls_scores * obj_scores[:, None]
        scores, labels = scores_per_class.max(dim=1)

        keep = scores > score_threshold

        pred_boxes = pred_boxes[keep]
        scores = scores[keep]
        labels = labels[keep]

        if len(pred_boxes) > 0:
            keep_nms = nms(pred_boxes, scores, nms_threshold)

            pred_boxes = pred_boxes[keep_nms]
            scores = scores[keep_nms]
            labels = labels[keep_nms]

        batch_predictions.append({
            "boxes": pred_boxes.detach().cpu(),
            "scores": scores.detach().cpu(),
            "labels": labels.detach().cpu()
        })

    return batch_predictions

Ниже определена вспомогательная функция для валидации качества. Можете использовать `Runner.validate`. Важное уточнение, ей нужен метод для фильтрации предсказаний. Можете тоже скопировать его из семинара, если он у вас не менялся.

In [ ]:
!pip install torchmetrics

In [ ]:
from torchmetrics.detection import MeanAveragePrecision

@torch.no_grad()
def validate(model, dataloader, filter_predictions_func, box_format="xyxy", device="cpu", score_threshold=0.1, nms_threshold=0.5):
    model.eval()

    metric = MeanAveragePrecision(box_format=box_format, iou_type="bbox")

    for images, targets in tqdm(dataloader, desc="Running validation", leave=False):
        images = images.to(device)

        outputs = model(images)
        predicts = filter_predictions_func(outputs, score_threshold, nms_threshold)

        targets_cpu = []
        for target in targets:
            targets_cpu.append({
                "boxes": target["boxes"].detach().cpu(),
                "labels": target["labels"].detach().cpu()
            })

        metric.update(predicts, targets_cpu)

    return metric.compute()["map"].item()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = Detector(num_classes=NUM_CLASSES, fpn_channels=128, unfreeze_last=2).to(device)

criterion = DetectionLoss(alpha=6.0, beta=1.0, topk=13)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)

num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for images, targets in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}"):
        images = images.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    map_score = validate(
        model=model,
        dataloader=test_loader,
        filter_predictions_func=filter_predictions,
        device=device,
        score_threshold=0.1,
        nms_threshold=0.5
    )

    print(f"Epoch {epoch + 1}: loss = {avg_loss:.4f}, mAP = {map_score:.4f}")

device: cpu
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 40.1MB/s]


Epoch 1/5:   0%|          | 0/58 [00:00<?, ?it/s]